## Challenge Five: Aero Alerts — Automated Weather Alerts for Airports

### Install dependencies

In [24]:
# BigQuery client + db-dtypes (to_dataframe). requests ships with Colab already.
%pip install --quiet --upgrade google-cloud-bigquery db-dtypes

In [25]:
from google.cloud import bigquery
import requests

### Variables

In [26]:
import os

# --- Diagnostic: shows what each detection method returns ---
print("=== Project detection diagnostic ===")
print("env GOOGLE_CLOUD_PROJECT:", os.environ.get("GOOGLE_CLOUD_PROJECT"))
print("env GCP_PROJECT         :", os.environ.get("GCP_PROJECT"))
try:
    import google.auth
    _creds, _proj = google.auth.default()
    print("google.auth project     :", _proj)
except Exception as e:
    print("google.auth failed      :", e)
print("=" * 36)


def detect_project_id():
    # 1. Explicit env var wins (lets anyone override without editing code)
    env = os.environ.get("GOOGLE_CLOUD_PROJECT") or os.environ.get("GCP_PROJECT")
    if env:
        return env
    # 2. Ask Application Default Credentials what project we're running under
    try:
        import google.auth
        _, project = google.auth.default()
        if project:
            return project
    except Exception:
        pass
    # 3. Last resort: query the metadata server (works on GCP runtimes)
    try:
        import urllib.request
        req = urllib.request.Request(
            "http://metadata.google.internal/computeMetadata/v1/project/project-id",
            headers={"Metadata-Flavor": "Google"},
        )
        return urllib.request.urlopen(req, timeout=2).read().decode()
    except Exception:
        return None


class Config:
    """Immutable run config. `project_id` is auto-detected from the runtime.
    Reads the source CSV straight from the public GCS bucket (no copy needed)."""

    def __init__(
        self,
        project_id=None,                      # None -> auto-detect from the runtime
        dataset="aero_alerts",
        location="US",                        # must match the GCS bucket region
        source_uri="gs://labs.roitraining.com/data-to-ai-workshop/airports.csv",
        raw_table_name="airports",
        large_view_name="us_large_airports",
        forecast_table_name="airport_forecasts",
        alert_table_name="airport_alerts",
        connection_name="gemini_conn",
        model_name="gemini_model",
        endpoint="gemini-2.5-flash",
        # NWS requires a descriptive User-Agent with a contact.
        nws_user_agent="aero-alerts (phanindra.jallipalli@zionclouds.com)",
    ):
        resolved = project_id or detect_project_id()
        if not resolved:
            raise ValueError(
                "Could not determine project_id. Pass it explicitly: "
                "Config(project_id='my-project')"
            )
        # bypass our own __setattr__ block during construction
        for k, v in {
            "project_id": resolved, "dataset": dataset, "location": location,
            "source_uri": source_uri, "raw_table_name": raw_table_name,
            "large_view_name": large_view_name,
            "forecast_table_name": forecast_table_name,
            "alert_table_name": alert_table_name,
            "connection_name": connection_name, "model_name": model_name,
            "endpoint": endpoint, "nws_user_agent": nws_user_agent,
        }.items():
            object.__setattr__(self, k, v)

    def __setattr__(self, name, value):
        raise AttributeError(f"Config is immutable; can't set {name!r}")

    @property
    def raw_table(self):
        return f"{self.project_id}.{self.dataset}.{self.raw_table_name}"

    @property
    def large_view(self):
        return f"{self.project_id}.{self.dataset}.{self.large_view_name}"

    @property
    def forecast_table(self):
        return f"{self.project_id}.{self.dataset}.{self.forecast_table_name}"

    @property
    def alert_table(self):
        return f"{self.project_id}.{self.dataset}.{self.alert_table_name}"

    @property
    def model(self):
        return f"{self.project_id}.{self.dataset}.{self.model_name}"

    @property
    def connection(self):
        return f"{self.project_id}.{self.location}.{self.connection_name}"


# --- Build the config ---
# Auto-detects on a GCP runtime. If detection fails, either:
#   - set os.environ["GOOGLE_CLOUD_PROJECT"] = "your-project" above, OR
#   - pass it directly: Config(project_id="your-project")
CFG = Config()

print("\nProject       :", CFG.project_id)
print("Raw table     :", CFG.raw_table)
print("Large view    :", CFG.large_view)
print("Forecast tbl  :", CFG.forecast_table)
print("Alert table   :", CFG.alert_table)
print("Connection    :", CFG.connection)
print("Model         :", CFG.model)

=== Project detection diagnostic ===
env GOOGLE_CLOUD_PROJECT: qwiklabs-gcp-01-5fe45b5e4e14
env GCP_PROJECT         : None
google.auth project     : qwiklabs-gcp-01-5fe45b5e4e14

Project       : qwiklabs-gcp-01-5fe45b5e4e14
Raw table     : qwiklabs-gcp-01-5fe45b5e4e14.aero_alerts.airports
Large view    : qwiklabs-gcp-01-5fe45b5e4e14.aero_alerts.us_large_airports
Forecast tbl  : qwiklabs-gcp-01-5fe45b5e4e14.aero_alerts.airport_forecasts
Alert table   : qwiklabs-gcp-01-5fe45b5e4e14.aero_alerts.airport_alerts
Connection    : qwiklabs-gcp-01-5fe45b5e4e14.US.gemini_conn
Model         : qwiklabs-gcp-01-5fe45b5e4e14.aero_alerts.gemini_model


### Logging

In [27]:
import logging, json, sys, uuid, datetime as dt

logger = logging.getLogger("aero_alerts")
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(h)

RUN_ID = str(uuid.uuid4())

def log_event(step: str, status: str, **kw):
    logger.info(json.dumps({"run_id": RUN_ID, "step": step, "status": status, **kw}))

log_event("init", "ok", started=dt.datetime.now(dt.timezone.utc).isoformat())

2026-06-03 15:36:42,528 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "init", "status": "ok", "started": "2026-06-03T15:36:42.528444+00:00"}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "init", "status": "ok", "started": "2026-06-03T15:36:42.528444+00:00"}


### Error handling — custom exceptions + step guard

In [28]:
import functools, traceback


class AeroAlertsError(Exception):
    """Base class for every error this pipeline raises on purpose.
    Carries the failing step, a friendly message, and the original cause."""
    def __init__(self, step: str, message: str, cause: Exception = None):
        self.step = step
        self.message = message
        self.cause = cause
        super().__init__(f"[{step}] {message}" + (f" (cause: {cause})" if cause else ""))


# --- Specific error types, one per failure domain --------------------------
class ConfigError(AeroAlertsError):        """Project / config could not be resolved."""
class DatasetError(AeroAlertsError):       """Dataset could not be created/accessed."""
class LoadError(AeroAlertsError):          """Loading the CSV from GCS into BigQuery failed."""
class ViewError(AeroAlertsError):          """Building the US-large-airports view failed."""
class ForecastFetchError(AeroAlertsError): """Calling the NWS API failed."""
class InsertError(AeroAlertsError):        """Stream-inserting rows into BigQuery failed."""
class ConnectionError_(AeroAlertsError):   """Creating the Vertex CLOUD_RESOURCE connection failed."""
class IAMError(AeroAlertsError):           """Granting the Vertex IAM role failed."""
class ModelError(AeroAlertsError):         """Creating the Gemini remote model failed."""
class GenerateError(AeroAlertsError):      """ML.GENERATE_TEXT alert generation failed."""
class QueryError(AeroAlertsError):         """A read/preview query failed."""


def guard(error_cls, friendly):
    """Decorator: wrap a pipeline method so any exception is logged as a
    structured error event and re-raised as `error_cls` with a clear message.
    `AeroAlertsError` raised inside is passed through unchanged (already typed)."""
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(self, *args, **kwargs):
            step = fn.__name__
            try:
                return fn(self, *args, **kwargs)
            except AeroAlertsError:
                raise  # already typed and logged at its origin
            except Exception as e:
                log_event(step, "error",
                          error_type=type(e).__name__,
                          message=str(e)[:300],
                          traceback=traceback.format_exc()[-800:])
                raise error_cls(step, friendly, e) from e
        return wrapper
    return decorator


def show_error(err: Exception):
    """Pretty-print any failure for the notebook reader."""
    if isinstance(err, AeroAlertsError):
        print("AERO ALERTS FAILED")
        print("  step    :", err.step)
        print("  problem :", err.message)
        if err.cause is not None:
            print("  cause   :", f"{type(err.cause).__name__}: {err.cause}")
    else:
        print("UNEXPECTED ERROR")
        print(" ", f"{type(err).__name__}: {err}")

### Pipeline: airports -> NWS forecasts -> Gemini alerts

In [29]:
import subprocess, time


class AeroAlertsPipeline:
    """Loads the airports CSV into BigQuery, narrows to US large airports,
    fetches each airport's current forecast from the National Weather Service,
    registers Gemini as a BigQuery remote model, and uses ML.GENERATE_TEXT to
    write a plain-language alert + severity for every airport."""

    # Narrow the 80k-row table to the proof-of-concept scope: US large airports.
    CREATE_VIEW_SQL = """
    CREATE OR REPLACE VIEW `{large_view}` AS
    SELECT
      ident, name, municipality, iso_region,
      latitude_deg, longitude_deg, elevation_ft, iata_code
    FROM `{raw_table}`
    WHERE type = 'large_airport'
      AND iso_country = 'US'
    """

    # Schema for the forecast rows we fetch from the NWS API (one row per airport).
    FORECAST_SCHEMA = [
        bigquery.SchemaField("ident",          "STRING",    mode="NULLABLE"),
        bigquery.SchemaField("name",           "STRING",    mode="NULLABLE"),
        bigquery.SchemaField("municipality",   "STRING",    mode="NULLABLE"),
        bigquery.SchemaField("iso_region",     "STRING",    mode="NULLABLE"),
        bigquery.SchemaField("latitude_deg",   "FLOAT64",   mode="NULLABLE"),
        bigquery.SchemaField("longitude_deg",  "FLOAT64",   mode="NULLABLE"),
        bigquery.SchemaField("forecast_period","STRING",    mode="NULLABLE"),
        bigquery.SchemaField("temperature",    "INT64",     mode="NULLABLE"),
        bigquery.SchemaField("temperature_unit","STRING",   mode="NULLABLE"),
        bigquery.SchemaField("wind_speed",     "STRING",    mode="NULLABLE"),
        bigquery.SchemaField("wind_direction", "STRING",    mode="NULLABLE"),
        bigquery.SchemaField("short_forecast", "STRING",    mode="NULLABLE"),
        bigquery.SchemaField("detailed_forecast","STRING",  mode="NULLABLE"),
        bigquery.SchemaField("fetched_at",     "TIMESTAMP", mode="NULLABLE"),
    ]

    # Registers Gemini as a BigQuery ML remote model via a CLOUD_RESOURCE connection.
    CREATE_MODEL_SQL = """
    CREATE OR REPLACE MODEL `{model}`
    REMOTE WITH CONNECTION `{connection}`
    OPTIONS (ENDPOINT = '{endpoint}')
    """

    # Builds a per-airport prompt from the forecast, calls Gemini inside SQL, and
    # writes the result into the alerts table. We ask for a single line of the form
    # SEVERITY | message so we can split it into two columns for Looker Studio.
    GENERATE_SQL = """
    CREATE OR REPLACE TABLE `{alert_table}` AS
    WITH generated AS (
      SELECT
        * EXCEPT(prompt, ml_generate_text_rai_result, ml_generate_text_status),
        ml_generate_text_llm_result AS raw_alert
      FROM ML.GENERATE_TEXT(
        MODEL `{model}`,
        (
          SELECT
            ident, name, municipality, iso_region,
            latitude_deg, longitude_deg,
            short_forecast, detailed_forecast, fetched_at,
            CONCAT(
              'You are an airport weather advisory officer for the FAA. ',
              'Using the forecast below, write ONE short, plain-language alert ',
              'for travelers and airport staff. Begin the line with a severity ',
              'label of NONE, ADVISORY, or WARNING, then a pipe, then the message. ',
              'Format exactly: SEVERITY | message. Keep the message under 40 words. ',
              'Use WARNING only for hazardous conditions (e.g. thunderstorms, ice, ',
              'high wind, fog, snow); ADVISORY for minor concerns; NONE if benign. ',
              'Airport: ', name, ' (', ident, '), ', municipality, ', ', iso_region,
              '. Period: ', forecast_period,
              '. Temperature: ', CAST(temperature AS STRING), ' ', temperature_unit,
              '. Wind: ', wind_speed, ' from ', wind_direction,
              '. Short forecast: ', short_forecast,
              '. Detailed forecast: ', detailed_forecast
            ) AS prompt
          FROM `{forecast_table}`
        ),
        STRUCT(
          0.3   AS temperature,         -- low = consistent, factual tone
          256   AS max_output_tokens,
          TRUE  AS flatten_json_output  -- clean ml_generate_text_llm_result column
        )
      )
    )
    SELECT
      ident, name, municipality, iso_region,
      latitude_deg, longitude_deg,
      short_forecast, fetched_at,
      -- Split "SEVERITY | message" into two columns; default to ADVISORY if unparseable.
      UPPER(COALESCE(NULLIF(TRIM(REGEXP_EXTRACT(raw_alert, r'^([^|]+)[|]')), ''), 'ADVISORY')) AS severity,
      TRIM(COALESCE(REGEXP_EXTRACT(raw_alert, r'[|](.*)$'), raw_alert)) AS alert_message,
      raw_alert
    FROM generated
    """

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.client = bigquery.Client(project=cfg.project_id, location=cfg.location)
        self.rows_in = 0
        self.airports = 0
        self.forecasts_fetched = 0
        self.fetch_errors = 0
        self.service_account = None

    # ---- BigQuery setup -------------------------------------------------
    @guard(DatasetError, "Could not create or access the BigQuery dataset. "
           "Check the project id and that you have BigQuery permissions.")
    def ensure_dataset(self):
        ds = bigquery.Dataset(f"{self.cfg.project_id}.{self.cfg.dataset}")
        ds.location = self.cfg.location
        self.client.create_dataset(ds, exists_ok=True)
        log_event("dataset_ready", "ok", dataset=self.cfg.dataset)

    @guard(LoadError, "Could not load airports.csv from Cloud Storage into BigQuery. "
           "Check the source_uri is reachable and the dataset location matches the bucket.")
    def load_raw(self):
        job_config = bigquery.LoadJobConfig(
            source_format=bigquery.SourceFormat.CSV,
            skip_leading_rows=1,
            autodetect=True,
            write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE,
        )
        job = self.client.load_table_from_uri(
            self.cfg.source_uri, self.cfg.raw_table, job_config=job_config
        )
        job.result()
        self.rows_in = self.client.get_table(self.cfg.raw_table).num_rows
        log_event("load_raw", "ok", rows=self.rows_in, job_id=job.job_id)

    @guard(ViewError, "Could not build the US large-airports view. "
           "Check that the airports table loaded and has the expected columns.")
    def create_large_view(self):
        sql = self.CREATE_VIEW_SQL.format(
            large_view=self.cfg.large_view, raw_table=self.cfg.raw_table
        )
        self.client.query(sql).result()
        self.airports = self.client.query(
            f"SELECT COUNT(*) AS c FROM `{self.cfg.large_view}`"
        ).to_dataframe().iloc[0]["c"]
        log_event("create_large_view", "ok", airports=int(self.airports))

    @guard(QueryError, "Could not read rows from the US large-airports view.")
    def preview_airports(self, n: int = 10):
        df = self.client.query(
            f"SELECT * FROM `{self.cfg.large_view}` ORDER BY ident LIMIT {n}"
        ).to_dataframe()
        log_event("preview_airports", "ok", rows=len(df))
        return df

    # ---- NWS forecast fetch (external API) ------------------------------
    def _nws_forecast(self, lat: float, lon: float):
        """Two-step NWS lookup: /points/{lat},{lon} -> forecast URL -> first period.
        Raises ForecastFetchError with a clear reason; the caller decides whether
        to skip that airport or stop."""
        headers = {"User-Agent": self.cfg.nws_user_agent, "Accept": "application/geo+json"}
        try:
            # 1) Resolve the grid point to get the forecast URL.
            pts = requests.get(
                f"https://api.weather.gov/points/{lat:.4f},{lon:.4f}",
                headers=headers, timeout=30,
            )
            pts.raise_for_status()
            forecast_url = pts.json()["properties"]["forecast"]
            # 2) Fetch the forecast; return the first (current) period.
            fc = requests.get(forecast_url, headers=headers, timeout=30)
            fc.raise_for_status()
            periods = fc.json()["properties"]["periods"]
            return periods[0] if periods else None
        except requests.exceptions.Timeout as e:
            raise ForecastFetchError("_nws_forecast", "NWS request timed out.", e) from e
        except requests.exceptions.HTTPError as e:
            code = getattr(e.response, "status_code", "?")
            raise ForecastFetchError(
                "_nws_forecast", f"NWS returned HTTP {code}.", e) from e
        except requests.exceptions.RequestException as e:
            raise ForecastFetchError(
                "_nws_forecast", "Network error calling the NWS API.", e) from e
        except (KeyError, ValueError) as e:
            raise ForecastFetchError(
                "_nws_forecast", "Unexpected NWS response shape (missing fields).", e) from e

    @guard(ForecastFetchError, "Could not set up or populate the forecast table. "
           "Check BigQuery permissions and NWS reachability.")
    def fetch_forecasts(self, sleep_seconds: float = 1.0):
        """Loop over US large airports, fetch each current forecast from NWS,
        and stream-insert one row per airport into the forecast table.
        Per-airport failures are logged and skipped so one bad airport never
        aborts the whole run; setup/insert failures stop the step."""
        # Recreate the table fresh each run so the schedule always reflects "now".
        self.client.delete_table(self.cfg.forecast_table, not_found_ok=True)
        table = bigquery.Table(self.cfg.forecast_table, schema=self.FORECAST_SCHEMA)
        self.client.create_table(table)
        log_event("forecast_table_ready", "ok", table=self.cfg.forecast_table)

        airports = self.client.query(
            f"SELECT * FROM `{self.cfg.large_view}` ORDER BY ident"
        ).to_dataframe()

        rows, now = [], dt.datetime.now(dt.timezone.utc).isoformat()
        for r in airports.itertuples(index=False):
            try:
                p = self._nws_forecast(r.latitude_deg, r.longitude_deg)
                if p is None:
                    self.fetch_errors += 1
                    log_event("fetch_forecast", "empty", ident=r.ident,
                              reason="NWS returned no forecast periods")
                    continue
                rows.append({
                    "ident": r.ident, "name": r.name,
                    "municipality": r.municipality, "iso_region": r.iso_region,
                    "latitude_deg": r.latitude_deg, "longitude_deg": r.longitude_deg,
                    "forecast_period": p.get("name"),
                    "temperature": p.get("temperature"),
                    "temperature_unit": p.get("temperatureUnit"),
                    "wind_speed": p.get("windSpeed"),
                    "wind_direction": p.get("windDirection"),
                    "short_forecast": p.get("shortForecast"),
                    "detailed_forecast": p.get("detailedForecast"),
                    "fetched_at": now,
                })
                self.forecasts_fetched += 1
            except ForecastFetchError as e:
                # Resilient: log this airport's reason and keep going.
                self.fetch_errors += 1
                log_event("fetch_forecast", "error", ident=r.ident, reason=e.message)
            time.sleep(sleep_seconds)

        if rows:
            errors = self.client.insert_rows_json(self.cfg.forecast_table, rows)
            if errors:
                log_event("fetch_forecasts", "insert_error", count=len(errors), sample=errors[:2])
                raise InsertError(
                    "fetch_forecasts",
                    f"BigQuery rejected {len(errors)} forecast row(s). First: {errors[:1]}")
        if self.forecasts_fetched == 0:
            raise ForecastFetchError(
                "fetch_forecasts",
                "No forecasts were fetched for any airport "
                f"({self.fetch_errors} failed). Check the NWS User-Agent and connectivity.")
        log_event("fetch_forecasts", "ok",
                  fetched=self.forecasts_fetched, errors=self.fetch_errors)

    @guard(QueryError, "Could not read rows from the forecast table.")
    def preview_forecasts(self, n: int = 10):
        df = self.client.query(f"""
        SELECT ident, name, forecast_period, temperature, temperature_unit,
               wind_speed, short_forecast
        FROM `{self.cfg.forecast_table}`
        ORDER BY ident LIMIT {n}
        """).to_dataframe()
        log_event("preview_forecasts", "ok", rows=len(df))
        return df

    # ---- Vertex / Gemini connection -------------------------------------
    @guard(ConnectionError_, "Could not create the Vertex CLOUD_RESOURCE connection or read "
           "its service account. Check that the BigQuery Connection API is enabled and bq is authed.")
    def ensure_connection(self):
        """Create the CLOUD_RESOURCE connection and capture its service account."""
        subprocess.run(
            ["bq", "mk", "--connection", f"--location={self.cfg.location}",
             f"--project_id={self.cfg.project_id}",
             "--connection_type=CLOUD_RESOURCE", self.cfg.connection_name],
            capture_output=True, text=True,
        )  # ignore "already exists"
        info = subprocess.run(
            ["bq", "show", "--format=json", "--connection", self.cfg.connection],
            capture_output=True, text=True,
        )
        if info.returncode != 0 or not info.stdout.strip():
            raise ConnectionError_(
                "ensure_connection",
                "`bq show` did not return the connection. "
                f"stderr: {info.stderr.strip()[:200]}")
        self.service_account = json.loads(info.stdout)["cloudResource"]["serviceAccountId"]
        log_event("ensure_connection", "ok", service_account=self.service_account)
        return self.service_account

    @guard(IAMError, "Could not grant the aiplatform.user role to the connection service "
           "account. You may lack resourcemanager.projects.setIamPolicy permission.")
    def grant_vertex_role(self, wait_seconds: int = 60):
        """Grant the connection SA the aiplatform.user role and let it propagate."""
        if not self.service_account:
            raise IAMError("grant_vertex_role",
                           "No connection service account; run ensure_connection() first.")
        res = subprocess.run(
            ["gcloud", "projects", "add-iam-policy-binding", self.cfg.project_id,
             f"--member=serviceAccount:{self.service_account}",
             "--role=roles/aiplatform.user", "--condition=None"],
            capture_output=True, text=True,
        )
        if res.returncode != 0:
            raise IAMError("grant_vertex_role",
                           f"gcloud IAM binding failed. stderr: {res.stderr.strip()[:200]}")
        log_event("grant_vertex_role", "ok", waiting=wait_seconds)
        time.sleep(wait_seconds)

    @guard(ModelError, "Could not create the Gemini remote model. Check the connection has the "
           "Vertex role (allow ~60s to propagate) and that the endpoint name is valid.")
    def create_model(self):
        sql = self.CREATE_MODEL_SQL.format(
            model=self.cfg.model, connection=self.cfg.connection,
            endpoint=self.cfg.endpoint,
        )
        self.client.query(sql).result()
        log_event("create_model", "ok", model=self.cfg.model_name, endpoint=self.cfg.endpoint)

    # ---- Generate alerts ------------------------------------------------
    @guard(GenerateError, "ML.GENERATE_TEXT failed. Check the model exists, the forecast table "
           "is populated, and the connection still has Vertex AI access.")
    def generate_alerts(self):
        sql = self.GENERATE_SQL.format(
            alert_table=self.cfg.alert_table, model=self.cfg.model,
            forecast_table=self.cfg.forecast_table,
        )
        job = self.client.query(sql)
        job.result()
        rows = self.client.get_table(self.cfg.alert_table).num_rows
        log_event("generate_alerts", "ok", rows=rows, job_id=job.job_id)

    @guard(QueryError, "Could not read rows from the alerts table.")
    def preview_alerts(self, n: int = 30):
        df = self.client.query(f"""
        SELECT ident, name, municipality, iso_region, severity, alert_message
        FROM `{self.cfg.alert_table}`
        ORDER BY
          CASE severity WHEN 'WARNING' THEN 0 WHEN 'ADVISORY' THEN 1 ELSE 2 END,
          ident
        LIMIT {n}
        """).to_dataframe()
        log_event("preview_alerts", "ok", rows=len(df))
        return df

    # ---- One-shot orchestration (used by the scheduler) -----------------
    def run(self):
        """Full refresh. Idempotent: safe to re-run on a schedule.
        Creates the connection + model only if they don't already exist yet."""
        self.ensure_dataset()
        self.load_raw()
        self.create_large_view()
        self.fetch_forecasts()
        self.ensure_connection()
        # Only grant + create the model when it isn't there yet (saves ~60s per run).
        try:
            self.client.get_model(self.cfg.model)
            log_event("create_model", "exists", model=self.cfg.model_name)
        except Exception:
            self.grant_vertex_role()
            self.create_model()
        self.generate_alerts()
        log_event("run", "ok", rows_in=self.rows_in, airports=int(self.airports),
                  forecasts=self.forecasts_fetched, errors=self.fetch_errors)

### Setup: dataset + load airports from Cloud Storage

In [30]:
try:
    pipeline = AeroAlertsPipeline(CFG)
    pipeline.ensure_dataset()
    pipeline.load_raw()
    print(f"Loaded {pipeline.rows_in:,} rows into {CFG.raw_table}")
except Exception as e:
    show_error(e)

2026-06-03 15:36:43,021 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "dataset_ready", "status": "ok", "dataset": "aero_alerts"}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "dataset_ready", "status": "ok", "dataset": "aero_alerts"}


2026-06-03 15:36:47,837 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "load_raw", "status": "ok", "rows": 82893, "job_id": "a54080d0-89e0-40f1-ba27-19ea3612e3f2"}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "load_raw", "status": "ok", "rows": 82893, "job_id": "a54080d0-89e0-40f1-ba27-19ea3612e3f2"}


Loaded 82,893 rows into qwiklabs-gcp-01-5fe45b5e4e14.aero_alerts.airports


### Narrow to US large airports (proof-of-concept scope)

In [31]:
try:
    pipeline.create_large_view()
    print(f"{int(pipeline.airports)} US large airports in {CFG.large_view}")
except Exception as e:
    show_error(e)

2026-06-03 15:36:50,944 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "create_large_view", "status": "ok", "airports": 71}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "create_large_view", "status": "ok", "airports": 71}


71 US large airports in qwiklabs-gcp-01-5fe45b5e4e14.aero_alerts.us_large_airports


### Preview the airports

In [32]:
try:
    display(pipeline.preview_airports())
except Exception as e:
    show_error(e)

2026-06-03 15:36:53,234 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "preview_airports", "status": "ok", "rows": 10}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "preview_airports", "status": "ok", "rows": 10}


,ident,name,municipality,iso_region,latitude_deg,longitude_deg,elevation_ft,iata_code
0,KABQ,Albuquerque International Sunport,Albuquerque,US-NM,35.039976,-106.608925,5355,ABQ
1,KADW,Joint Base Andrews,Camp Springs,US-MD,38.810799,-76.866997,280,ADW
2,KATL,Hartsfield Jackson Atlanta International Airport,Atlanta,US-GA,33.636700,-84.428101,1026,ATL
3,KAUS,Austin Bergstrom International Airport,Austin,US-TX,30.197535,-97.662015,542,AUS
4,KBDL,Bradley International Airport,Hartford,US-CT,41.938510,-72.688066,173,BDL
5,KBNA,Nashville International Airport,Nashville,US-TN,36.124500,-86.678200,599,BNA
6,KBOS,Logan International Airport,Boston,US-MA,42.361970,-71.007900,20,BOS
7,KBUF,Buffalo Niagara International Airport,Buffalo,US-NY,42.940498,-78.732201,728,BUF
8,KBWI,Baltimore/Washington International Thurgood Ma...,Baltimore,US-MD,39.175400,-76.668297,146,BWI
9,KCLE,Cleveland Hopkins International Airport,Cleveland,US-OH,41.411701,-81.849800,791,CLE


### Fetch current forecasts from the National Weather Service

Two-step NWS lookup per airport: `/points/{lat},{lon}` returns the forecast URL,
then the forecast URL returns the periods; we keep the first (current) period.
A 1-second pause between airports keeps us within the NWS rate limits, so this
takes roughly half a minute for ~30 airports. Per-airport failures are logged and
skipped; the step only fails if *no* airport could be fetched.

In [33]:
try:
    pipeline.fetch_forecasts()
    print(f"Forecasts fetched : {pipeline.forecasts_fetched}")
    print(f"Fetch errors      : {pipeline.fetch_errors}")
except Exception as e:
    show_error(e)

2026-06-03 15:36:53,657 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "forecast_table_ready", "status": "ok", "table": "qwiklabs-gcp-01-5fe45b5e4e14.aero_alerts.airport_forecasts"}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "forecast_table_ready", "status": "ok", "table": "qwiklabs-gcp-01-5fe45b5e4e14.aero_alerts.airport_forecasts"}


2026-06-03 15:38:31,222 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "fetch_forecasts", "status": "ok", "fetched": 71, "errors": 0}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "fetch_forecasts", "status": "ok", "fetched": 71, "errors": 0}


Forecasts fetched : 71
Fetch errors      : 0


### Preview the forecasts

In [34]:
try:
    display(pipeline.preview_forecasts())
except Exception as e:
    show_error(e)

2026-06-03 15:38:33,486 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "preview_forecasts", "status": "ok", "rows": 10}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "preview_forecasts", "status": "ok", "rows": 10}


,ident,name,forecast_period,temperature,temperature_unit,wind_speed,short_forecast
0,KABQ,Albuquerque International Sunport,Today,83,F,5 to 10 mph,Partly Sunny then Chance Showers And Thunderst...
1,KADW,Joint Base Andrews,Today,81,F,8 mph,Sunny
2,KATL,Hartsfield Jackson Atlanta International Airport,Today,78,F,10 mph,Sunny
3,KAUS,Austin Bergstrom International Airport,Today,90,F,0 to 10 mph,Partly Sunny then Chance Showers And Thunderst...
4,KBDL,Bradley International Airport,Today,84,F,1 to 5 mph,Sunny
5,KBNA,Nashville International Airport,Today,83,F,0 to 5 mph,Sunny
6,KBOS,Logan International Airport,Today,79,F,2 to 8 mph,Sunny
7,KBUF,Buffalo Niagara International Airport,Today,78,F,5 to 12 mph,Sunny
8,KBWI,Baltimore/Washington International Thurgood Ma...,Today,83,F,7 mph,Sunny
9,KCLE,Cleveland Hopkins International Airport,Today,74,F,2 to 7 mph,Sunny


### Connect to Vertex AI — create connection + grant role

In [35]:
try:
    pipeline.ensure_connection()
    print("Connection service account:", pipeline.service_account)
    pipeline.grant_vertex_role()
    print("IAM role granted and propagated.")
except Exception as e:
    show_error(e)

2026-06-03 15:38:38,352 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "ensure_connection", "status": "ok", "service_account": "bqcx-299593570351-o8m6@gcp-sa-bigquery-condel.iam.gserviceaccount.com"}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "ensure_connection", "status": "ok", "service_account": "bqcx-299593570351-o8m6@gcp-sa-bigquery-condel.iam.gserviceaccount.com"}


Connection service account: bqcx-299593570351-o8m6@gcp-sa-bigquery-condel.iam.gserviceaccount.com
2026-06-03 15:38:40,007 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "grant_vertex_role", "status": "ok", "waiting": 60}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "grant_vertex_role", "status": "ok", "waiting": 60}


IAM role granted and propagated.


### Add Gemini to the project — CREATE MODEL

In [36]:
try:
    pipeline.create_model()
    print(f"Remote model {CFG.model} created, pointing to {CFG.endpoint}.")
except Exception as e:
    show_error(e)

2026-06-03 15:39:42,642 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "create_model", "status": "ok", "model": "gemini_model", "endpoint": "gemini-2.5-flash"}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "create_model", "status": "ok", "model": "gemini_model", "endpoint": "gemini-2.5-flash"}


Remote model qwiklabs-gcp-01-5fe45b5e4e14.aero_alerts.gemini_model created, pointing to gemini-2.5-flash.


### Generate alerts in SQL — ML.GENERATE_TEXT

In [37]:
try:
    pipeline.generate_alerts()
    print(f"Alerts written to {CFG.alert_table}.")
except Exception as e:
    show_error(e)

2026-06-03 15:39:56,991 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "generate_alerts", "status": "ok", "rows": 71, "job_id": "13274222-71d8-42fb-8ffe-281019a2fa8b"}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "generate_alerts", "status": "ok", "rows": 71, "job_id": "13274222-71d8-42fb-8ffe-281019a2fa8b"}


Alerts written to qwiklabs-gcp-01-5fe45b5e4e14.aero_alerts.airport_alerts.


### Review the generated alerts (most severe first)

In [38]:
try:
    display(pipeline.preview_alerts())
except Exception as e:
    show_error(e)

2026-06-03 15:39:59,205 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "preview_alerts", "status": "ok", "rows": 30}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "preview_alerts", "status": "ok", "rows": 30}


,ident,name,municipality,iso_region,severity,alert_message
0,KFLL,Fort Lauderdale Hollywood International Airport,Fort Lauderdale,US-FL,WARNING,Thunderstorms and showers are likely throughou...
1,KMIA,Miami International Airport,Miami,US-FL,WARNING,Thunderstorms with heavy rain expected today. ...
2,KMSY,Louis Armstrong New Orleans International Airport,New Orleans,US-LA,WARNING,Thunderstorms likely today with gusts up to 30...
3,KPBI,Palm Beach International Airport,West Palm Beach,US-FL,WARNING,"Thunderstorms likely today at KPBI, with winds..."
4,KABQ,Albuquerque International Sunport,Albuquerque,US-NM,ADVISORY,Chance of afternoon showers and thunderstorms ...
5,KAUS,Austin Bergstrom International Airport,Austin,US-TX,ADVISORY,Chance of showers and thunderstorms this after...
6,KDEN,Denver International Airport,Denver,US-CO,ADVISORY,KDEN: Chance of afternoon showers and thunders...
7,KDFW,Dallas Fort Worth International Airport,Dallas-Fort Worth,US-TX,ADVISORY,Slight chance of showers and thunderstorms aft...
8,KIAH,George Bush Intercontinental Houston Airport,Houston,US-TX,ADVISORY,Chance of showers and thunderstorms today. Exp...
9,KJAX,Jacksonville International Airport,Jacksonville,US-FL,ADVISORY,Expect sunny skies with a northeast wind aroun...


### One-shot run (what the scheduler executes)

`run()` does the full refresh end to end and is idempotent: the dataset, view,
forecast table and alert table are all recreated/replaced each time, and the
Gemini connection + model are reused if they already exist. Run this cell once
manually to confirm it works, then schedule the notebook (next section).

On failure it prints a clear `step / problem / cause` summary and re-raises, so a
scheduled run is marked **failed** (not silently green) — important for GenAIOps.

In [39]:
try:
    pipeline = AeroAlertsPipeline(CFG)
    pipeline.run()
    print("Refresh complete.")
    display(pipeline.preview_alerts())
except Exception as e:
    show_error(e)
    raise  # re-raise so the scheduler records the run as failed

2026-06-03 15:39:59,721 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "dataset_ready", "status": "ok", "dataset": "aero_alerts"}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "dataset_ready", "status": "ok", "dataset": "aero_alerts"}


2026-06-03 15:40:06,420 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "load_raw", "status": "ok", "rows": 82893, "job_id": "c45d4d8f-3f10-480d-acbc-2f2bbd29127c"}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "load_raw", "status": "ok", "rows": 82893, "job_id": "c45d4d8f-3f10-480d-acbc-2f2bbd29127c"}


2026-06-03 15:40:10,283 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "create_large_view", "status": "ok", "airports": 71}


...

2026-06-03 15:41:56,654 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "run", "status": "ok", "rows_in": 82893, "airports": 71, "forecasts": 71, "errors": 0}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "run", "status": "ok", "rows_in": 82893, "airports": 71, "forecasts": 71, "errors": 0}


Refresh complete.
2026-06-03 15:41:58,737 | INFO | {"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "preview_alerts", "status": "ok", "rows": 30}


INFO:aero_alerts:{"run_id": "324288ef-a5d4-4911-a160-4a5097479ee8", "step": "preview_alerts", "status": "ok", "rows": 30}


,ident,name,municipality,iso_region,severity,alert_message
0,KFLL,Fort Lauderdale Hollywood International Airport,Fort Lauderdale,US-FL,WARNING,Thunderstorms likely today. Expect potential d...
1,KMIA,Miami International Airport,Miami,US-FL,WARNING,Thunderstorms and heavy rain likely today. Exp...
2,KMSY,Louis Armstrong New Orleans International Airport,New Orleans,US-LA,WARNING,Thunderstorms likely today with gusts up to 30...
3,KPBI,Palm Beach International Airport,West Palm Beach,US-FL,WARNING,"Thunderstorms and showers likely today, with w..."
4,KABQ,Albuquerque International Sunport,Albuquerque,US-NM,ADVISORY,"Chance of afternoon showers and thunderstorms,..."
5,KATL,Hartsfield Jackson Atlanta International Airport,Atlanta,US-GA,ADVISORY,Sunny skies and pleasant temperatures today. E...
6,KAUS,Austin Bergstrom International Airport,Austin,US-TX,ADVISORY,Chance of showers and thunderstorms after 1 PM.
7,KDEN,Denver International Airport,Denver,US-CO,ADVISORY,KDEN: Chance of afternoon showers and thunders...
8,KDFW,Dallas Fort Worth International Airport,Dallas-Fort Worth,US-TX,ADVISORY,Slight chance of showers and thunderstorms aft...
9,KIAH,George Bush Intercontinental Houston Airport,Houston,US-TX,ADVISORY,Chance of showers and thunderstorms today. Exp...
